# ШАГ 1. НАСТРОЙКА ОКРУЖЕНИЯ, ИМПОРТЫ, КОНФИГУРАЦИЯ И АППАРАТНАЯ ДИАГНОСТИКА

### 1. Структурированный импорт библиотек

In [ ]:
# Системные модули
import os
import sys
import random
import psutil

# Цифровая обработка сигналов
import numpy as np
import h5py
import scipy.signal as signal     # Для фильтрации и классических допплеровских алгоритмов
import scipy.fft as fft           # Для быстрого преобразования Фурье

# Глубокое обучение (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Визуализация данных
import matplotlib.pyplot as plt

### 2. Интеграция с Google Drive и определение путей

In [ ]:
try:
    from google.colab import drive
    # Проверяем, не смонтирован ли диск ранее, чтобы избежать лишних диалоговых окон
    if not os.path.exists('/content/drive/MyDrive'):
        print("[ИНФО] Монтирование Google Диска...")
        drive.mount('/content/drive')
    else:
        print("[ИНФО] Google Диск уже смонтирован.")
except ImportError:
    print("[ПРЕДУПРЕЖДЕНИЕ] Среда выполнения отличается от Google Colab. "
          "Монтирование Google Диска пропущено.")

In [ ]:
# Определение глобальных констант путей для проекта
DRIVE_ROOT = '/content/drive/MyDrive/UNet3D_Clutter'
DATA_DIR = os.path.join(DRIVE_ROOT, 'dataset')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
LOGS_DIR = os.path.join(DRIVE_ROOT, 'logs')

In [ ]:
# Автоматическое создание недостающих директорий в облаке
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

In [ ]:
print(f"[ИНФО] Базовый путь проекта: {DRIVE_ROOT}")
print(f"[ИНФО] Путь к датасету:       {DATA_DIR}")
print(f"[ИНФО] Путь к чекпоинтам:     {CHECKPOINTS_DIR}")
print(f"[ИНФО] Путь к логам обучения: {LOGS_DIR}\n")

### 3. Единый блок глобальной конфигурации (CONFIG)

In [ ]:
CONFIG = {
    'SEED': 42,
    'BATCH_SIZE': 8,              # С учетом ограничений памяти Tesla T4 (16 Гб VRAM)
    'NUM_EPOCHS': 100,
    'LEARNING_RATE': 1e-4,
    'WEIGHT_DECAY': 1e-5,         # Параметр регуляризации для оптимизатора AdamW
    'IN_CHANNELS': 2,             # I (вещественный) и Q (мнимый) каналы
    'OUT_CHANNELS': 2,            # I и Q каналы на выходе
    'BASE_FEATURES': 32,          # Число каналов на первом уровне энкодера
    'NUM_ENCODER_LEVELS': 3,       # Глубина U-Net (влияет на требования к размеру T/X/Y)
    'T_DIM': 32,                  # Длина временного ансамбля (ось медленного времени)
    'X_DIM': 64,                  # Латеральный размер патча
    'Y_DIM': 64,                  # Аксиальный размер патча (глубина)
    'LAMBDA_SPECTRAL': 0.1,       # Весовой коэффициент спектрального лосса (Spatio-Spectral Loss)
    'TRAIN_RATIO': 0.80,          # Доля обучающей выборки
    'VAL_RATIO': 0.10,            # Доля валидационной выборки (оставшиеся 0.1 — тест)
    'PATIENCE': 10,               # Количество эпох без улучшений для Early Stopping
    'NUM_WORKERS': 2              # Важно: при создании DataLoader необходимо передать
                                  # worker_init_fn для фиксации seed в каждом дочернем процессе
}

### 4. Полная фиксация генераторов случайных чисел (Reproducibility)

In [ ]:
"""
Фиксирует все возможные генераторы случайных чисел для обеспечения
стабильности вычислений и воспроизводимости результатов.
"""
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Настройка детерминированного режима работы библиотеки CuDNN
    # Внимание: deterministic = True может незначительно замедлить 3D-свертки,
    # но это необходимо для обеспечения полной воспроизводимости при проверке
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[ИНФО] Генераторы случайных чисел зафиксированы. SEED = {seed}")

In [ ]:
set_reproducibility(CONFIG['SEED'])

### 5. Инициализация целевого устройства (Device Allocation)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[ИНФО] Целевое устройство для вычислений: {device}\n")

### 6. Комплексная аппаратная диагностика ресурсов

In [ ]:
def run_hardware_diagnostics():
    """
    Собирает и выводит в консоль детальную информацию о версиях библиотек,
    объеме системной оперативной памяти (RAM) и физической видеопамяти (VRAM).
    """
    import scipy

    print("==================================================")
    print("       КОМПЛЕКСНАЯ ДИАГНОСТИКА РЕСУРСОВ СРЕДЫ     ")
    print("==================================================")

    print("ВЕРСИИ БИБЛИОТЕК:")
    print(f"  PyTorch версия:     {torch.__version__}")
    if torch.cuda.is_available():
        print(f"  CUDA версия (torch): {torch.version.cuda}")
    print(f"  NumPy версия:       {np.__version__}")
    print(f"  h5py версия:        {h5py.__version__}")
    print(f"  SciPy версия:       {scipy.__version__}")
    print("-" * 50)

    # 1. Диагностика системной оперативной памяти (RAM)
    virtual_mem = psutil.virtual_memory()
    total_ram_gb = virtual_mem.total / (1024 ** 3)
    used_ram_gb = virtual_mem.used / (1024 ** 3)
    free_ram_gb = virtual_mem.available / (1024 ** 3)

    print("СИСТЕМНАЯ ПАМЯТЬ (System RAM):")
    print(f"  Всего памяти:       {total_ram_gb:.2f} GB")
    print(f"  Занято:             {used_ram_gb:.2f} GB ({virtual_mem.percent}%)")
    print(f"  Доступно:           {free_ram_gb:.2f} GB")
    print("-" * 50)

    # 2. Диагностика видеопамяти графического ускорителя (VRAM)
    print(f"ВЫЧИСЛИТЕЛЬНОЕ УСТРОЙСТВО: {device}")

    if device.type == 'cuda':
        gpu_name = torch.cuda.get_device_name(0)

        # Запрос физического состояния видеопамяти напрямую у драйвера CUDA
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)

        total_vram_gb = total_bytes / (1024 ** 3)
        free_vram_gb = free_bytes / (1024 ** 3)
        allocated_vram_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
        reserved_vram_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)

        print(f"ГРАФИЧЕСКИЙ УСКОРИТЕЛЬ (VRAM):")
        print(f"  Модель GPU:                 {gpu_name}")
        print(f"  Общий объем VRAM (физ.):    {total_vram_gb:.2f} GB")
        print(f"  Свободно VRAM (физ. CUDA):  {free_vram_gb:.2f} GB")
        print(f"  Занято аллокатором PyTorch: {allocated_vram_gb:.2f} GB")
        print(f"  Зарезервировано PyTorch:    {reserved_vram_gb:.2f} GB")
    else:
        print("[ПРЕДУПРЕЖДЕНИЕ] Видеокарта с поддержкой CUDA не обнаружена.")
        print("Вычисления будут производиться на системном процессоре (CPU), "
              "что существенно замедлит обучение 3D-сверток.")
    print("==================================================")

In [ ]:
# Запуск аппаратной диагностики после инициализации
run_hardware_diagnostics()

# Шаг 2: Разработка конвейера загрузки данных (HDF5 PyTorch Dataset & DataLoader)

In [ ]:
import glob
import tempfile

### 1. Класс набора данных DopplerDataset(Dataset)

In [ ]:
class DopplerDataset(Dataset):
    """
    Класс набора данных для загрузки комплексных IQ-данных ультразвуковой
    допплерографии из файлов HDF5 (.h5).
    """
    def __init__(self, filepaths):
        """
        Инициализирует Dataset списком путей к файлам.
        """
        self.filepaths = filepaths

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        """
        Загружает один сэмпл данных, выполняет численно устойчивую
        локальную нормализацию и возвращает тензоры в формате PyTorch.
        """
        filepath = self.filepaths[idx]

        # Локальное открытие файла внутри __getitem__ для обеспечения потокобезопасности
        with h5py.File(filepath, 'r') as f:
            input_data = np.array(f['input'], dtype=np.float32)
            target_data = np.array(f['target'], dtype=np.float32)

        # Поскольку каналы I и Q находятся в нулевом измерении (C=2, индексы 0 и 1),
        # мы извлекаем их по первому индексу массива.
        I_in = input_data[0]
        Q_in = input_data[1]

        # Расчет амплитуды огибающей входной смеси
        amplitude = np.sqrt(I_in**2 + Q_in**2)
        a_max = np.max(amplitude)

        # Численно устойчивое ограничение снизу (клиппинг знаменателя)
        a_max = np.maximum(a_max, 1e-9)

        # Нормализация с сохранением относительного энергетического баланса
        input_data = input_data / a_max
        target_data = target_data / a_max

        # Конвертация в тензоры PyTorch.
        input_tensor = torch.from_numpy(input_data).float()
        target_tensor = torch.from_numpy(target_data).float()

        return input_tensor, target_tensor

### 2. Функция создания загрузчиков get_dataloaders(config)

In [ ]:
def seed_worker(worker_id):
    """
    Инициализирует генераторы случайных чисел в дочерних потоках DataLoader.
    Предотвращает дублирование псевдослучайных последовательностей.
    """
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
def get_dataloaders(config):
    """
    Сканирует директорию с данными, динамически разделяет файлы на
    обучающую, валидационную и тестовую выборки и возвращает объекты DataLoader.
    """
    # Сканирование директории и получение списка файлов
    search_pattern = os.path.join(config['DATA_DIR'], '*.h5')
    filepaths = sorted(glob.glob(search_pattern))

    if len(filepaths) == 0:
        raise ValueError(f"[ОШИБКА] В директории {config['DATA_DIR']} не найдено файлов .h5")

    print(f"[ИНФО] Всего обнаружено файлов .h5: {len(filepaths)}")

    # Перемешивание списка файлов для воспроизводимости
    rng = np.random.RandomState(config['SEED'])
    filepaths_shuffled = list(filepaths)
    rng.shuffle(filepaths_shuffled)

    # Динамический расчет индексов разделения (Single Source of Truth)
    num_files = len(filepaths_shuffled)
    train_end = int(num_files * config['TRAIN_RATIO'])
    val_end = train_end + int(num_files * config['VAL_RATIO'])

    train_files = filepaths_shuffled[:train_end]
    val_files = filepaths_shuffled[train_end:val_end]
    test_files = filepaths_shuffled[val_end:]

    print(f"[ИНФО] Обучающая выборка (Train):     {len(train_files)} файлов")
    print(f"[ИНФО] Валидационная выборка (Val):  {len(val_files)} файлов")
    print(f"[ИНФО] Тестовая выборка (Test):      {len(test_files)} файлов")

    # Создание экземпляров DopplerDataset
    train_dataset = DopplerDataset(train_files)
    val_dataset = DopplerDataset(val_files)
    test_dataset = DopplerDataset(test_files) if len(test_files) > 0 else None

    # Инициализация системного генератора PyTorch для воспроизводимого батчинга
    g = torch.Generator()
    g.manual_seed(config['SEED'])

    use_cuda = torch.cuda.is_available()

    # DataLoader для обучающей выборки
    train_loader = DataLoader(
        train_dataset,
        batch_size=config['BATCH_SIZE'],
        shuffle=True,
        num_workers=config['NUM_WORKERS'],
        pin_memory=use_cuda,
        persistent_workers=True if config['NUM_WORKERS'] > 0 else False,
        worker_init_fn=seed_worker,
        generator=g,
        drop_last=True
    )

    # DataLoader для валидационной выборки
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['BATCH_SIZE'],
        shuffle=False,
        num_workers=config['NUM_WORKERS'],
        pin_memory=use_cuda,
        persistent_workers=True if config['NUM_WORKERS'] > 0 else False,
        worker_init_fn=seed_worker,
        generator=g
    )

    # DataLoader для тестовой выборки (при наличии)
    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(
            test_dataset,
            batch_size=config['BATCH_SIZE'],
            shuffle=False,
            num_workers=config['NUM_WORKERS'],
            pin_memory=use_cuda,
            persistent_workers=True if config['NUM_WORKERS'] > 0 else False,
            worker_init_fn=seed_worker,
            generator=g
        )

    return train_loader, val_loader, test_loader

# Шаг 3. Программная реализация архитектуры 3D U-Net (Model Definition)

### 1. Модульная структура классов (Компоненты сети)

In [ ]:
class DoubleConv3d(nn.Module):
    """
    Базовый сверточный блок 3D U-Net.
    Выполняет два последовательных сверточных преобразования 3x3x3 с
    последующей нормализацией InstanceNorm3D и функцией активации LeakyReLU.
    Не изменяет пространственно-временную размерность тензора за счет padding=1.
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv3d, self).__init__()
        self.double_conv = nn.Sequential(
            # Первая свертка: bias=False, так как за ней сразу следует InstanceNorm3d
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels, affine=True),
            nn.LeakyReLU(negative_slope=0.01, inplace=True),  # inplace=True существенно экономит VRAM

            # Вторая свертка
            nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels, affine=True),
            nn.LeakyReLU(negative_slope=0.01, inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

In [ ]:
class EncoderBlock(nn.Module):
    """
    Блок энкодера (путь сжатия).
    Применяет блок двойной 3D-свертки, после чего выполняет операцию
    макспулинга MaxPool3d для уменьшения пространственно-временного разрешения в 2 раза.
    """
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.double_conv = DoubleConv3d(in_channels, out_channels)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        # Сохраняем промежуточное состояние до пулинга для skip-connection
        skip_connection = self.double_conv(x)
        # Понижаем пространственно-временное разрешение
        pooled = self.pool(skip_connection)
        return skip_connection, pooled

In [ ]:
class DecoderBlock(nn.Module):
    """
    Блок декодера (путь восстановления).
    Выполняет операцию транспонированной 3D-свертки (апсэмплинг),
    конкатенирует полученный тензор со skip-connection от соответствующего уровня энкодера
    и пропускает объединенный результат через блок двойной 3D-свертки.
    """
    def __init__(self, in_channels, skip_channels, out_channels):
        super(DecoderBlock, self).__init__()
        # Транспонированная свертка восстанавливает разрешение по осям T, X, Y в 2 раза.
        # bias=False исключает избыточность, так как после слияния следует блок нормализации.
        self.up = nn.ConvTranspose3d(
            in_channels,
            skip_channels,
            kernel_size=2,
            stride=2,
            bias=False
        )
        # Входной размер для DoubleConv3d равен сумме каналов апсэмпленного слоя и skip-connection
        self.double_conv = DoubleConv3d(skip_channels * 2, out_channels)

    def forward(self, x, skip_connection):
        # Увеличиваем разрешение в 2 раза
        x_up = self.up(x)
        # Объединяем тензоры по оси каналов (dim=1)
        x_concat = torch.cat([x_up, skip_connection], dim=1)
        # Пропускаем через DoubleConv3d
        return self.double_conv(x_concat)

### 2. Главный класс сети ClutterFilter3DUNet(nn.Module)

In [ ]:
class ClutterFilter3DUNet(nn.Module):
    """
    Полноразмерная нейросеть 3D U-Net для адаптивной фильтрации допплеровских сигналов.
    Проектируется по концепции Software-Defined Medical Device для интеграции
    в платформы реального времени через TensorRT.
    """
    def __init__(self, in_channels=2, out_channels=2, base_features=32):
        super(ClutterFilter3DUNet, self).__init__()

        # --- Encoder ---
        self.enc1 = EncoderBlock(in_channels, base_features)                 # 2 -> 32
        self.enc2 = EncoderBlock(base_features, base_features * 2)           # 32 -> 64
        self.enc3 = EncoderBlock(base_features * 2, base_features * 4)       # 64 -> 128

        # --- Bottleneck ---
        # На самом глубоком уровне пространственное разрешение минимально, макспулинг не применяется
        self.bottleneck = DoubleConv3d(base_features * 4, base_features * 8) # 128 -> 256

        # --- Decoder ---
        self.dec3 = DecoderBlock(base_features * 8, base_features * 4, base_features * 4) # 256 -> 128
        self.dec2 = DecoderBlock(base_features * 4, base_features * 2, base_features * 2) # 128 -> 64
        self.dec1 = DecoderBlock(base_features * 2, base_features, base_features)         # 64 -> 32

        # --- Финальный выходной слой ---
        # Свертка 1x1x1 осуществляет проекцию признаков в исходное пространство I и Q каналов.
        # bias=True необходим для корректного восстановления абсолютных значений амплитуды
        self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1, stride=1, padding=0, bias=True)

        # Автоматический запуск явной инициализации весов
        self.apply(self._init_weights)

    # Явная инициализация весов
    def _init_weights(self, m):
        """
        Инициализирует веса слоев согласно распределению Кайминга
        для обеспечения численной стабильности градиентов на старте обучения.
        """
        if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
            # Инициализация Кайминга, оптимизированная под LeakyReLU(0.01)
            nn.init.kaiming_normal_(m.weight, a=0.01, nonlinearity='leaky_relu')
            if m.bias is not None:
                # Обнуление смещений при их наличии
                nn.init.constant_(m.bias, 0.0)

        elif isinstance(m, nn.InstanceNorm3d):
            # Инициализация параметров масштабирования (gamma) единицами,
            # а параметров сдвига (beta) нулями
            if m.weight is not None:
                nn.init.constant_(m.weight, 1.0)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        # Проход по пути сжатия (сохраняем skip-connections на каждом шаге)
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)

        # Прохождение через узкое горлышко (Bottleneck)
        x = self.bottleneck(x)

        # Проход по пути восстановления с конкатенацией признаков энкодера
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Финальное линейное преобразование в 2 выходных канала (I и Q)
        return self.out_conv(x)

# ШАГ 4. РЕАЛИЗАЦИЯ КОМПЛЕКСНЫХ И СПЕКТРАЛЬНЫХ ФУНКЦИЙ ПОТЕРЬ

### 1. Архитектура и математическая модель функций потерь

In [ ]:
class NormalizedComplexMSELoss(nn.Module):
    """
    Нормированный комплексный лосс (NormalizedComplexMSELoss).
    Оценивает относительную ошибку восстановления комплексного IQ-сигнала.
    """
    def __init__(self, eps=1e-4):  # eps = 1e-4 соответствует физическому порогу шума в -40 дБ
        super(NormalizedComplexMSELoss, self).__init__()
        self.eps = eps

    def forward(self, pred, target):
        # Принудительное приведение к float32 выводит расчет из-под режима AMP (Float16),
        # гарантируя точность вычислений и защиту от переполнения градиентов
        pred = pred.float()
        target = target.float()

        # Входные тензоры: [B, 2, T, X, Y]
        pred_I, pred_Q = pred[:, 0], pred[:, 1]
        target_I, target_Q = target[:, 0], target[:, 1]

        # Вычисляем квадрат ошибки: [B, T, X, Y]
        error = (pred_I - target_I)**2 + (pred_Q - target_Q)**2

        # Вычисляем квадрат мощности эталона: [B, T, X, Y]
        power = target_I**2 + target_Q**2

        # Пообъектное усреднение по осям (T, X, Y) -> получаем [B]
        mean_error = torch.mean(error, dim=(1, 2, 3))
        mean_power = torch.mean(power, dim=(1, 2, 3))

        # Вычисляем отношение пообъектно и усредняем по батчу
        loss = torch.mean(mean_error / (mean_power + self.eps))
        return loss


class SpatioSpectralLoss(nn.Module):
    """
    Совмещенный спектральный лосс (SpatioSpectralLoss).
    Объединяет пространственный и логарифмический спектральный лоссы.
    """
    def __init__(self, lambda_spec=0.1, eps=1e-4):
        super(SpatioSpectralLoss, self).__init__()
        self.lambda_spec = lambda_spec
        self.eps = eps
        self.spatial_loss_fn = NormalizedComplexMSELoss(eps=eps)

    def forward(self, pred, target):
        # Принудительное приведение к float32 гарантирует, что БПФ и логарифмирование
        # будут выполнены в полноценном FP32 без вызова предупреждений ComplexHalf и CPU fallback
        pred = pred.float()
        target = target.float()

        # 1. Расчет пространственной ошибки
        L_spatial = self.spatial_loss_fn(pred, target)

        # 2. Перевод каналов I и Q в комплексное представление для БПФ
        pred_complex = torch.complex(pred[:, 0], pred[:, 1])
        target_complex = torch.complex(target[:, 0], target[:, 1])

        # 3. Одномерное БПФ по оси медленного времени (dim=1 соответствует оси T)
        pred_fft = torch.fft.fft(pred_complex, dim=1)
        target_fft = torch.fft.fft(target_complex, dim=1)

        # 4. Численно устойчивый расчет амплитудного спектра
        pred_amp = torch.sqrt(pred_fft.real**2 + pred_fft.imag**2 + 1e-12)
        target_amp = torch.sqrt(target_fft.real**2 + target_fft.imag**2 + 1e-12)

        # 5. Перевод амплитуд в логарифмический масштаб (дБ) с порогом шума -40 дБ
        pred_log = torch.log10(pred_amp + self.eps)
        target_log = torch.log10(target_amp + self.eps)

        # 6. Расчет спектральной ошибки (MSE в логарифмическом спектре)
        L_spectral = torch.mean((pred_log - target_log)**2)

        # 7. Формирование совмещенного лосса
        L_joint = L_spatial + self.lambda_spec * L_spectral

        return L_joint, L_spatial

# ШАГ 5. ЦИКЛ ОБУЧЕНИЯ, ВАЛИДАЦИИ И ЧЕКПОИНТИНГА

In [ ]:
import os
import time
import json
import shutil
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

### 1. Модульная структура компонентов (Архитектурный план)

In [ ]:
class EarlyStopping:
    """
    Класс ранней остановки (EarlyStopping).
    Отслеживает динамику изменения валидационного лосса и сигнализирует
    о необходимости досрочного завершения обучения при переобучении модели.
    """
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < (self.best_loss - self.min_delta):
            # Зафиксировано значимое улучшение — сбрасываем счетчик ожидания
            self.best_loss = val_loss
            self.counter = 0
        else:
            # Улучшений нет — инкрементируем счетчик простоя
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
def save_checkpoint(state, is_best, local_dir, drive_dir):
    """
    Сохранение чекпоинта.
    Сначала записывает файлы на локальный диск сессии, а затем асинхронно
    копирует их на Google Диск.
    """
    os.makedirs(local_dir, exist_ok=True)
    os.makedirs(drive_dir, exist_ok=True)

    local_latest_path = os.path.join(local_dir, 'checkpoint_latest.pth')
    local_best_path = os.path.join(local_dir, 'model_best.pth')

    # 1. Локальное сохранение состояния на быстрый SSD
    torch.save(state, local_latest_path)
    if is_best:
        shutil.copy2(local_latest_path, local_best_path)

    # 2. Перенос чекпоинтов на смонтированный Google Диск
    try:
        shutil.copy2(local_latest_path, os.path.join(drive_dir, 'checkpoint_latest.pth'))
        if is_best:
            shutil.copy2(local_best_path, os.path.join(drive_dir, 'model_best.pth'))
    except Exception as e:
        print(f"[ПРЕДУПРЕЖДЕНИЕ] Сбой синхронизации чекпоинта с Google Диском: {str(e)}")

In [ ]:
def load_checkpoint(model, optimizer, scheduler, scaler, drive_dir, local_dir, device):
    """
    Загрузка чекпоинта.
    Проверяет наличие сохраненных состояний на Google Диске, а при их отсутствии —
    в локальной директории. Восстанавливает веса, параметры оптимизаторов и логов.
    """
    drive_path = os.path.join(drive_dir, 'checkpoint_latest.pth')
    local_path = os.path.join(local_dir, 'checkpoint_latest.pth')

    path_to_load = None
    if os.path.exists(drive_path):
        path_to_load = drive_path
        print(f"[ИНФО] Обнаружен чекпоинт на Google Диске: {drive_path}")
    elif os.path.exists(local_path):
        path_to_load = local_path
        print(f"[ИНФО] Обнаружен локальный чекпоинт в сессии: {local_path}")

    if path_to_load is None:
        print("[ИНФО] Чекпоинты не найдены. Обучение начнется с нуля (Эпоха 1).")
        return 0, float('inf'), []

    # Чтение чекпоинта на целевое устройство (предотвращает утечки VRAM)
    checkpoint = torch.load(path_to_load, map_location=device)

    # Восстановление состояний компонентов графа
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if scaler is not None and 'scaler_state_dict' in checkpoint and checkpoint['scaler_state_dict'] is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    epoch = checkpoint['epoch']
    best_val_loss = checkpoint['best_val_loss']
    history = checkpoint.get('history', [])

    print(f"[ИНФО] Чекпоинт загружен. Возобновление обучения со следующей эпохи: {epoch + 2}. "
          f"Лучший Val Loss на данный момент: {best_val_loss:.6f}")

    return epoch + 1, best_val_loss, history

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device, config):
    """
    Основной тренировочный цикл train_model.
    Координирует фазы обучения, пообъектной валидации, изменения темпа обучения,
    ранней остановки и двойного резервного чекпоинтинга.
    """
    local_dir = '/content/local_checkpoints'
    drive_dir = config.get('CHECKPOINTS_DIR', CHECKPOINTS_DIR)
    logs_dir = config.get('LOGS_DIR', LOGS_DIR)

    os.makedirs(logs_dir, exist_ok=True)
    early_stopping = EarlyStopping(patience=config['PATIENCE'])

    # Определение поддержки аппаратного смешанного режима вычислений (AMP)
    use_amp = (device.type == 'cuda')
    # Инициализация GradScaler для масштабирования градиентов в PyTorch 2.x
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    # Попытка возобновить обучение из чекпоинта
    start_epoch, best_val_loss, history = load_checkpoint(
        model, optimizer, scheduler, scaler, drive_dir, local_dir, device
    )

    print(f"[ИНФО] Обучение запущено на: {device}. Использование смешанной точности (AMP): {use_amp}")

    for epoch in range(start_epoch, config['NUM_EPOCHS']):
        epoch_start_time = time.time()

        # --- ШАГ 1: Обучение модели ---
        model.train()
        train_loss_accum = 0.0
        train_steps = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            # set_to_none=True снижает накладные расходы на обращение к VRAM
            optimizer.zero_grad(set_to_none=True)

            # Прямой проход с автоматическим приведением типов FP16/FP32
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(inputs)
                loss, _ = criterion(outputs, targets)

            # Обратное распространение и шаг оптимизатора
            if use_amp:
                # Масштабирование градиентов для предотвращения недоучета малых шагов
                scaler.scale(loss).backward()
                # Размасштабирование перед клиппингом нормы градиентов
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                # Шаг оптимизатора и обновление масштаба
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            train_loss_accum += loss.item()
            train_steps += 1

        train_epoch_loss = train_loss_accum / train_steps if train_steps > 0 else 0.0

        # --- ШАГ 2: Валидация модели ---
        model.eval()
        val_loss_accum = 0.0
        val_steps = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)

                with torch.amp.autocast('cuda', enabled=use_amp):
                    outputs = model(inputs)
                    loss, _ = criterion(outputs, targets)

                val_loss_accum += loss.item()
                val_steps += 1

        val_epoch_loss = val_loss_accum / val_steps if val_steps > 0 else 0.0

        # --- ШАГ 3: Обновление планировщика и ранняя остановка ---
        # ReduceLROnPlateau принимает текущую метрику валидации в качестве аргумента
        scheduler.step(val_epoch_loss)
        early_stopping(val_epoch_loss)

        # Логирование показателей эпохи
        epoch_time = time.time() - epoch_start_time
        current_lr = optimizer.param_groups[0]['lr']

        epoch_metrics = {
            'epoch': epoch + 1,
            'train_loss': train_epoch_loss,
            'val_loss': val_epoch_loss,
            'lr': current_lr,
            'time': epoch_time
        }
        history.append(epoch_metrics)

        # Сохранение полной истории логов в формате JSON на Диск
        history_path = os.path.join(logs_dir, 'metrics_history.json')
        with open(history_path, 'w') as f:
            json.dump(history, f, indent=4)

        # --- ШАГ 4: Чекпоинтинг ---
        is_best = (val_epoch_loss < best_val_loss)
        if is_best:
            best_val_loss = val_epoch_loss

        state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if use_amp else None,
            'best_val_loss': best_val_loss,
            'history': history
        }

        save_checkpoint(state, is_best, local_dir, drive_dir)

        # Вывод консольного лога
        status_str = "[УЛУЧШЕНО! Лучший результат]" if is_best else ""
        print(f"Эпоха {epoch+1:03d}/{config['NUM_EPOCHS']:03d} | "
              f"Время: {epoch_time:.2f} сек | "
              f"Train Loss: {train_epoch_loss:.6f} | "
              f"Val Loss: {val_epoch_loss:.6f} | "
              f"LR: {current_lr:.2e} {status_str}")

        if early_stopping.early_stop:
            print(f"[ИНФО] Сработал триггер Early Stopping (Patience = {config['PATIENCE']}). "
                  f"Обучение завершено.")
            break

    return history

# ШАГ 6. ОЦЕНКА КАЧЕСТВА И ВИЗУАЛИЗАЦИЯ

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

### 1. Специфические метрики качества УЗДГ

In [ ]:
def calculate_cs(z_in, z_out, mask_clutter, eps=1e-12):
    """
    Эффективность подавления клаттера (Clutter Suppression, CS).
    Рассчитывает отношение мощностей входного и выходного сигналов в зоне
    неподвижных тканей (определяемой маской mask_clutter).
    Входные тензоры: [2, T, X, Y]
    """
    z_in = z_in.float()
    z_out = z_out.float()
    mask_clutter = mask_clutter.float()

    # Расчет мощности сигналов [T, X, Y]
    power_in = z_in[0]**2 + z_in[1]**2
    power_out = z_out[0]**2 + z_out[1]**2

    # Суммирование энергии по осям времени и пространства внутри маски клаттера
    # Путем добавления оси T к маске [X, Y] -> [1, X, Y]
    sum_in = torch.sum(power_in * mask_clutter.unsqueeze(0))
    sum_out = torch.sum(power_out * mask_clutter.unsqueeze(0))

    # Вычисление коэффициента подавления в децибелах (дБ)
    cs = 10 * torch.log10(sum_in / (sum_out + eps))
    return cs.item()

In [ ]:
def calculate_bsp(z_target, z_out, mask_vessel, eps=1e-12):
    """
    Сохранение сигнала крови (Blood Signal Preservation, BSP).
    Вычисляет локальное отношение полезного сигнала к ошибке (SER) в децибелах
    непосредственно внутри зоны сосуда (определяемой маской mask_vessel).
    Входные тензоры: [2, T, X, Y]
    """
    z_target = z_target.float()
    z_out = z_out.float()
    mask_vessel = mask_vessel.float()

    # Расчет эталонной мощности и мощности ошибки восстановления [T, X, Y]
    power_target = z_target[0]**2 + z_target[1]**2
    error_diff = (z_out[0] - z_target[0])**2 + (z_out[1] - z_target[1])**2

    # Суммирование энергии внутри маски сосуда
    sum_target = torch.sum(power_target * mask_vessel.unsqueeze(0))
    sum_error = torch.sum(error_diff * mask_vessel.unsqueeze(0))

    # Расчет сохранности полезного сигнала
    bsp = 10 * torch.log10(sum_target / (sum_error + eps))
    return bsp.item()

In [ ]:
def calculate_mve(z_target, z_out, mask_vessel):
    """
    Ошибка оценки средней скорости (Mean Velocity Error, MVE).
    Рассчитывает среднее абсолютное отклонение (MAE) фазовых сдвигов (в радианах)
    по методу автокорреляции Касаи внутри зоны сосуда.
    Входные тензоры: [2, T, X, Y]
    """
    z_target = z_target.float()
    z_out = z_out.float()
    mask_vessel = mask_vessel.float()

    # Перевод сигналов в комплексное представление для спектрального анализа
    z_target_c = torch.complex(z_target[0], z_target[1])
    z_out_c = torch.complex(z_out[0], z_out[1])

    # Автокорреляция первого порядка по временной оси T (ось 0) -> R(x,y)
    R_target = torch.sum(z_target_c[1:] * torch.conj(z_target_c[:-1]), dim=0)
    R_out = torch.sum(z_out_c[1:] * torch.conj(z_out_c[:-1]), dim=0)

    # Оценка фазовых сдвигов (средняя скорость) в радианах [-pi, pi]
    V_target = torch.angle(R_target)
    V_out = torch.angle(R_out)

    # Вычисление когерентной разности фаз с учетом циклического сдвига (wrap-around)
    diff = torch.angle(torch.exp(1j * (V_out - V_target)))
    abs_diff = torch.abs(diff)

    # Усреднение ошибки исключительно внутри маски сосуда
    mve = torch.sum(abs_diff * mask_vessel) / (torch.sum(mask_vessel) + 1e-8)
    return mve.item()

### 2. Схема комплексной визуализации

In [ ]:
def plot_doppler_results(z_in, z_target, z_out, mask_vessel, prf=1000, save_path=None):
    """
    Строит 4-панельный дашборд
    Входные тензоры: [2, T, X, Y]
    """
    # Перенос тензоров на CPU для визуализации в Matplotlib
    z_in = z_in.cpu().float()
    z_target = z_target.cpu().float()
    z_out = z_out.cpu().float()

    T = z_in.shape[1]

    # Создание общего окна дашборда
    fig = plt.figure(figsize=(18, 12))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.2)

    # --------------------------------------------------------------------------
    # Панель 1: Энергетический допплер (Power Doppler Map)
    # --------------------------------------------------------------------------
    # Разбиваем первый квадрант на 3 горизонтальных субплота
    gs_pd = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[0, 0], wspace=0.15)
    ax_pd_in = fig.add_subplot(gs_pd[0, 0])
    ax_pd_tgt = fig.add_subplot(gs_pd[0, 1])
    ax_pd_out = fig.add_subplot(gs_pd[0, 2])

    # Вычисление мощности в дБ [X, Y]
    p_in = 10 * np.log10(torch.mean(z_in[0]**2 + z_in[1]**2, dim=0).numpy() + 1e-8)
    p_tgt = 10 * np.log10(torch.mean(z_target[0]**2 + z_target[1]**2, dim=0).numpy() + 1e-8)
    p_out = 10 * np.log10(torch.mean(z_out[0]**2 + z_out[1]**2, dim=0).numpy() + 1e-8)

    # Нормализуем шкалу по пиковому уровню входа для репрезентативного сравнения
    max_power = np.max(p_in)
    p_in_norm = p_in - max_power
    p_tgt_norm = p_tgt - max_power
    p_out_norm = p_out - max_power

    # Отрисовка 2D энергетических карт.
    # Транспонирование .T переводит форму из [X, Y] в [Y, X], размещая глубину Y на вертикальной оси.
    # Поведение по умолчанию (origin='upper') оставляет апертуру датчика (глубина 0) на верхней грани.
    im_in = ax_pd_in.imshow(p_in_norm.T, cmap='viridis', vmin=-45, vmax=0, origin='upper')
    ax_pd_in.set_title("Вход (Смесь)")
    ax_pd_in.set_ylabel("Глубина сканирования Y")
    ax_pd_in.set_xlabel("Луч сканирования X")

    im_tgt = ax_pd_tgt.imshow(p_tgt_norm.T, cmap='viridis', vmin=-45, vmax=0, origin='upper')
    ax_pd_tgt.set_title("Эталон (Кровь)")
    ax_pd_tgt.axis('off')

    im_out = ax_pd_out.imshow(p_out_norm.T, cmap='viridis', vmin=-45, vmax=0, origin='upper')
    ax_pd_out.set_title("Выход 3D U-Net")
    ax_pd_out.axis('off')

    # Общая горизонтальная цветовая панель
    cbar_pd = fig.colorbar(im_out, ax=[ax_pd_in, ax_pd_tgt, ax_pd_out], orientation='horizontal', pad=0.1, shrink=0.8)
    cbar_pd.set_label("Относительная мощность сигнала (дБ)")

    # --------------------------------------------------------------------------
    # Панель 2: Пространственно-временная спектрограмма
    # --------------------------------------------------------------------------
    # Разбиваем второй квадрант на 2 субплота (Вход vs Выход)
    gs_spec = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[0, 1], wspace=0.15)
    ax_spec_in = fig.add_subplot(gs_spec[0, 0])
    ax_spec_out = fig.add_subplot(gs_spec[0, 1])

    # Фиксируем аксиальный срез Y_c ровно посередине сосуда (индекс 32)
    Y_c = 32
    slice_in_c = torch.complex(z_in[0, :, :, Y_c], z_in[1, :, :, Y_c])
    slice_out_c = torch.complex(z_out[0, :, :, Y_c], z_out[1, :, :, Y_c])

    # 1D БПФ по оси медленного времени (dim=0) с частотным сдвигом.
    fft_in = torch.fft.fftshift(torch.fft.fft(slice_in_c, dim=0), dim=0)
    fft_out = torch.fft.fftshift(torch.fft.fft(slice_out_c, dim=0), dim=0)

    spec_in_db = 20 * torch.log10(torch.abs(fft_in) + 1e-4).numpy()
    spec_out_db = 20 * torch.log10(torch.abs(fft_out) + 1e-4).numpy()

    # Задание осей координат: лучи X по абсциссе, спектральная частота от -PRF/2 до PRF/2 по ординате
    extent = [0, z_in.shape[2] - 1, -prf/2, prf/2]

    # Для спектрограмм по-прежнему важен origin='lower', чтобы отрицательные частоты были внизу
    ax_spec_in.imshow(spec_in_db, cmap='jet', extent=extent, aspect='auto', origin='lower')
    ax_spec_in.set_title("Спектр Входа (Клаттер на 0 Гц)")
    ax_spec_in.set_xlabel("Координата луча X")
    ax_spec_in.set_ylabel("Допплеровский сдвиг (Гц)")

    ax_spec_out.imshow(spec_out_db, cmap='jet', extent=extent, aspect='auto', origin='lower')
    ax_spec_out.set_title("Спектр Выхода (Профиль потока)")
    ax_spec_out.set_xlabel("Координата луча X")
    ax_spec_out.get_yaxis().set_visible(False) # Скрываем ось Y для дублирующего плота

    # --------------------------------------------------------------------------
    # Панель 3: Спектр мощности в пикселе (Single-Pixel Power Spectrum)
    # --------------------------------------------------------------------------
    ax_single_pixel = fig.add_subplot(gs[1, 0])

    # Фиксируем центральный пиксель кровотока
    X_c, Y_c = 32, 32
    freq_axis = np.linspace(-prf/2, prf/2, T)

    # Расчет спектров трех состояний в децибелах
    vec_in = torch.complex(z_in[0, :, X_c, Y_c], z_in[1, :, X_c, Y_c])
    ps_in = 10 * np.log10(np.abs(torch.fft.fftshift(torch.fft.fft(vec_in)).numpy())**2 + 1e-4)

    vec_tgt = torch.complex(z_target[0, :, X_c, Y_c], z_target[1, :, X_c, Y_c])
    ps_tgt = 10 * np.log10(np.abs(torch.fft.fftshift(torch.fft.fft(vec_tgt)).numpy())**2 + 1e-4)

    vec_out = torch.complex(z_out[0, :, X_c, Y_c], z_out[1, :, X_c, Y_c])
    ps_out = 10 * np.log10(np.abs(torch.fft.fftshift(torch.fft.fft(vec_out)).numpy())**2 + 1e-4)

    # Отрисовка графиков
    ax_single_pixel.plot(freq_axis, ps_in, label='Вход', color='gray', alpha=0.7, linestyle='--')
    ax_single_pixel.plot(freq_axis, ps_tgt, label='Эталон (Кровь)', color='green', linewidth=2)
    ax_single_pixel.plot(freq_axis, ps_out, label='Выход 3D U-Net', color='red', linewidth=1.5)

    ax_single_pixel.set_title(f"Спектр мощности в пикселе ({X_c}, {Y_c})")
    ax_single_pixel.set_xlabel("Частота Допплера (Гц)")
    ax_single_pixel.set_ylabel("Амплитуда спектра (дБ)")
    ax_single_pixel.grid(True, linestyle=':', alpha=0.6)
    ax_single_pixel.legend()

    # --------------------------------------------------------------------------
    # Панель 4: Цветовой допплер скоростей
    # --------------------------------------------------------------------------
    ax_color_doppler = fig.add_subplot(gs[1, 1])

    # Автокорреляция по Касаи первого порядка для выхода сети
    z_out_c = torch.complex(z_out[0], z_out[1])
    R = torch.sum(z_out_c[1:] * torch.conj(z_out_c[:-1]), dim=0).numpy()
    V_velocity = np.angle(R)

    # Принудительное зануление хаотичного фазового шума вне сосуда:
    # если средняя мощность выхода в пикселе ниже -35 дБ относительно пика — заменяем значение на NaN (белый фон)
    power_out_db = 10 * np.log10(torch.mean(z_out[0]**2 + z_out[1]**2, dim=0).numpy() + 1e-8)
    power_out_db_norm = power_out_db - np.max(power_out_db)

    masked_V = np.copy(V_velocity)
    masked_V[power_out_db_norm < -15] = np.nan

    # Задание цветовой схемы и маскирование "плохих" значений белым цветом.
    # Транспонирование .T выстраивает верную анатомическую ориентацию [Y, X]
    cmap_doppler = plt.get_cmap('RdBu_r').copy()
    cmap_doppler.set_bad(color='white')

    im_color = ax_color_doppler.imshow(masked_V.T, cmap=cmap_doppler, vmin=-np.pi, vmax=np.pi, origin='upper')
    ax_color_doppler.set_title("Карта допплеровских скоростей (Касаи)")
    ax_color_doppler.set_ylabel("Глубина сканирования Y")
    ax_color_doppler.set_xlabel("Луч сканирования X")

    cbar_color = fig.colorbar(im_color, ax=ax_color_doppler, pad=0.05, shrink=0.8)
    cbar_color.set_label("Сдвиг фазы за импульс (рад)")

    # Сохранение или вывод на экран
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"[ИНФО] Дашборд успешно экспортирован по пути: {save_path}")
    plt.show()
    plt.close(fig)

# ШАГ 7. ПАКЕТНОЕ ОБУЧЕНИЕ, ТЕСТОВЫЙ ИНФЕРЕНС И АНАЛИТИКА

In [ ]:
import os
import glob
import h5py
import json
import torch
import shutil
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1. Запуск полного обучения

In [ ]:
def run_full_training(model, train_loader, val_loader, optimizer, scheduler, criterion, device, config):
    """
    Запускает цикл обучения модели 3D U-Net на полном числе эпох
    и возвращает историю изменения метрик.
    """
    print("\n[ФАЗА А] Запуск полного цикла обучения модели...")
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
        config=config
    )
    print("[ФАЗА А] Обучение успешно завершено. Веса сохранены на Google Диск.")
    return history

### 2. Тестовый инференс и статистический анализ

In [ ]:
def run_test_inference_and_analysis(model, dataset, device, config):
    """
    Выполняет итерационный проход по тестовой выборке, рассчитывает допплеровские
    метрики для каждого сэмпла и формирует сводный отчет в формате Pandas DataFrame.
    """
    print("\n[ФАЗА Б/В] Запуск тестового инференса и количественной оценки...")
    model.eval()

    results_list = []

    for idx in range(len(dataset)):
        # Считываем сэмпл напрямую из датасета
        x, y = dataset[idx]
        filepath = dataset.filepaths[idx]
        filename = os.path.basename(filepath)

        # Динамическая оценка маски сосуда по энергии эталонного сигнала крови (Target)
        # Рассчитываем мощность в каждом пикселе [X, Y]
        target_power = torch.mean(y[0]**2 + y[1]**2, dim=0)
        max_power = torch.max(target_power)

        # Клинический порог отсечки -35 дБ для локализации зоны кровотока
        mask_vessel = (target_power > (max_power * (10 ** -3.5))).float()
        mask_clutter = 1.0 - mask_vessel

        # Проверяем, содержит ли файл предопределенный 'filtered' набор (для dry run тестов)
        # Это позволяет имитировать работу идеальной модели на искусственных данных
        use_mock_filtered = False
        with h5py.File(filepath, 'r') as f:
            if 'filtered' in f:
                use_mock_filtered = True
                z_out = torch.from_numpy(np.array(f['filtered'])).float()

        if not use_mock_filtered:
            # Боевой инференс через веса 3D U-Net
            # Добавляем размерность батча [B, C, T, X, Y] -> [1, 2, T, X, Y]
            x_batch = x.unsqueeze(0).to(device)
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    out_batch = model(x_batch)
            # Убираем размерность батча и переносим на CPU
            z_out = out_batch.squeeze(0).cpu()

        # Вычисление трех допплеровских метрик (из Шага 6)
        cs_val = calculate_cs(x, z_out, mask_clutter)
        bsp_val = calculate_bsp(y, z_out, mask_vessel)
        mve_val = calculate_mve(y, z_out, mask_vessel)

        results_list.append({
            'File_Name': filename,
            'Clutter_Suppression_dB': cs_val,
            'Blood_Preservation_dB': bsp_val,
            'Mean_Velocity_Error_rad': mve_val
        })

    # Формирование сводного Pandas отчета
    df_results = pd.DataFrame(results_list)

    # Сохранение результатов в формате CSV на Google Диск
    logs_dir = config.get('LOGS_DIR', LOGS_DIR)
    os.makedirs(logs_dir, exist_ok=True)
    csv_path = os.path.join(logs_dir, 'test_results_summary.csv')
    df_results.to_csv(csv_path, index=False)

    print(f"[ИНФО] Сводная таблица метрик успешно экспортирована: {csv_path}")

    # Расчет и вывод интегральных статистических показателей (Среднее ± Ст.Откл)
    mean_cs, std_cs = df_results['Clutter_Suppression_dB'].mean(), df_results['Clutter_Suppression_dB'].std()
    mean_bsp, std_bsp = df_results['Blood_Preservation_dB'].mean(), df_results['Blood_Preservation_dB'].std()
    mean_mve, std_mve = df_results['Mean_Velocity_Error_rad'].mean(), df_results['Mean_Velocity_Error_rad'].std()

    print("\n" + "="*50)
    print("      ИНТЕГРАЛЬНЫЕ КЛИНИЧЕСКИЕ ПОКАЗАТЕЛИ (ТЕСТ)  ")
    print("="*50)
    print(f"  CS  (Подавление клаттера): {mean_cs:.1f} ± {std_cs:.1f} dB")
    print(f"  BSP (Сохранение крови):    {mean_bsp:.1f} ± {std_bsp:.1f} dB")
    print(f"  MVE (Фазовая ошибка):       {mean_mve:.4f} ± {std_mve:.4f} rad")
    print("="*50)

    return df_results

### 3. Экспорт характерных примеров для графической части

In [ ]:
def export_representative_cases(model, dataset, df_results, device, config):
    """
    Сортирует тестовые результаты по качеству восстановления кровотока (BSP)
    и экспортирует 4-панельные дашборды для лучшего, медианного и худшего случаев.
    """
    print("\n[ФАЗА Г] Поиск и графический экспорт характерных клинических примеров...")

    # Сортировка таблицы по качеству сохранения сигнала крови (BSP)
    df_sorted = df_results.sort_values(by='Blood_Preservation_dB').reset_index(drop=True)
    num_samples = len(df_sorted)

    # Выделение пограничных случаев
    worst_info = df_sorted.iloc[0]                          # Минимальный BSP
    median_info = df_sorted.iloc[num_samples // 2]          # Медианный BSP
    best_info = df_sorted.iloc[-1]                          # Максимальный BSP

    cases = {
        'best': (best_info, 'result_best.png'),
        'median': (median_info, 'result_median.png'),
        'worst': (worst_info, 'result_worst.png')
    }

    logs_dir = config.get('LOGS_DIR', LOGS_DIR)

    for case_name, (info, filename) in cases.items():
        fname = info['File_Name']
        bsp_val = info['Blood_Preservation_dB']

        print(f"  - Экспорт {case_name.upper()} случая: {fname} (BSP = {bsp_val:.2f} дБ)")

        # Поиск сэмпла в наборе данных по имени файла
        idx = -1
        for i, path in enumerate(dataset.filepaths):
            if os.path.basename(path) == fname:
                idx = i
                break

        if idx == -1:
            print(f"[ПРЕДУПРЕЖДЕНИЕ] Файл {fname} не найден в структуре тестового датасета.")
            continue

        x, y = dataset[idx]
        filepath = dataset.filepaths[idx]

        # Получение выхода (повторяем инференс-логику Фазы Б)
        use_mock_filtered = False
        with h5py.File(filepath, 'r') as f:
            if 'filtered' in f:
                use_mock_filtered = True
                z_out = torch.from_numpy(np.array(f['filtered'])).float()

        if not use_mock_filtered:
            x_batch = x.unsqueeze(0).to(device)
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    out_batch = model(x_batch)
            z_out = out_batch.squeeze(0).cpu()

        # Генерация маски сосуда
        target_power = torch.mean(y[0]**2 + y[1]**2, dim=0)
        max_power = torch.max(target_power)
        mask_vessel = (target_power > (max_power * (10 ** -3.5))).float()

        # Отрисовка и экспорт дашборда на Диск
        save_plot_path = os.path.join(logs_dir, filename)
        plot_doppler_results(
            z_in=x,
            z_target=y,
            z_out=z_out,
            mask_vessel=mask_vessel,
            prf=1000,
            save_path=save_plot_path
        )

# Шаг 8. ЗАПУСК КОНВЕЙЕРА

In [ ]:
CONFIG['DATA_DIR'] = DATA_DIR
CONFIG['CHECKPOINTS_DIR'] = CHECKPOINTS_DIR
CONFIG['LOGS_DIR'] = LOGS_DIR

In [ ]:
# Создание загрузчиков данных для реального датасета из Google Диска
train_loader, val_loader, test_loader = get_dataloaders(CONFIG)

In [ ]:
# Инициализация аппаратных компонентов
model = ClutterFilter3DUNet(
    in_channels=CONFIG['IN_CHANNELS'],
    out_channels=CONFIG['OUT_CHANNELS'],
    base_features=CONFIG['BASE_FEATURES']
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY']
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=3  # Шаг уменьшения LR при застое на val_loss
)

criterion = SpatioSpectralLoss(lambda_spec=CONFIG['LAMBDA_SPECTRAL']).to(device)

In [ ]:
# Запуск процесса обучения
# Если сессия прервется, при повторном запуске код автоматически найдет
# файл 'checkpoint_latest.pth' и продолжит обучение с сохраненной эпохи.
history = run_full_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    config=CONFIG
)

In [ ]:
# Тестовый инференс, расчет метрик и экспорт дашбордов
test_dataset = test_loader.dataset

In [ ]:
# Оценка качества фильтрации на тестовой выборке с расчетом средних CS, BSP, MVE
df_results = run_test_inference_and_analysis(
    model=model,
    dataset=test_dataset,
    device=device,
    config=CONFIG
)

In [ ]:
# Нахождение лучшего, медианного и худшего случаев и экспорт графиков .png
export_representative_cases(
    model=model,
    dataset=test_dataset,
    df_results=df_results,
    device=device,
    config=CONFIG
)

# Шаг 9. СРАВНИТЕЛЬНЫЙ АНАЛИЗ

In [ ]:
import os
import glob
import random
import json
import shutil
import warnings
import tempfile
warnings.filterwarnings('ignore')

import numpy as np
import h5py
import pandas as pd
import scipy.signal as signal_proc
import matplotlib
matplotlib.use('Agg')   # Безголовый рендеринг для надёжности в Colab
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

### Шаг 1. Монтирование Google Drive

In [ ]:
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("[ИНФО] Монтирование Google Диска...")
        drive.mount('/content/drive')
    else:
        print("[ИНФО] Google Диск уже смонтирован.")
except ImportError:
    print("[ПРЕДУПРЕЖДЕНИЕ] Среда отличается от Google Colab. Drive не монтируется.")

### Шаг 2. Конфигурация сравнительного анализа

In [ ]:
COMPARE_CONFIG = {
    # --- Пути (должны совпадать с путями основного скрипта) ---
    'DATA_DIR':        '/content/drive/MyDrive/UNet3D_Clutter/dataset',
    'CHECKPOINTS_DIR': '/content/drive/MyDrive/UNet3D_Clutter/checkpoints',
    'OUTPUT_DIR':      '/content/drive/MyDrive/UNet3D_Clutter/comparison_results',

    # --- Параметры разбивки датасета (должны совпадать с обучением) ---
    'SEED':        42,
    'TRAIN_RATIO': 0.80,
    'VAL_RATIO':   0.10,

    # --- Параметры архитектуры модели ---
    'IN_CHANNELS':   2,
    'OUT_CHANNELS':  2,
    'BASE_FEATURES': 32,

    # --- Параметры классических фильтров ---
    'PRF':                 1000,   # Частота повторения импульсов [Гц]
    'FILTER_CUTOFF_RATIO': 0.10,   # Доля от PRF для всех классических фильтров
    'FIR_ORDER':           8,      # КИХ-фильтр: ограничение N≤8 из-за T=32
    'IIR_ORDER':           2,      # БИХ-фильтр Баттерворта
    'SVD_RANK':            2,      # SVD: число подавляемых клаттер-компонент

    # --- Режим данных ---
    'USE_MOCK_DATA':  False,
    'MOCK_SAMPLES':   20,

    # --- Порог маски сосуда (в дБ ниже пика мощности эталона) ---
    'VESSEL_MASK_THRESHOLD_DB': -35.0,

    # --- Визуализация ---
    'FIGURE_DPI': 120,
}

### Шаг 3. Инициализация устройства

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(COMPARE_CONFIG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"[ИНФО] Целевое устройство: {device}")
if device.type == 'cuda':
    print(f"[ИНФО] GPU: {torch.cuda.get_device_name(0)}  |  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

os.makedirs(COMPARE_CONFIG['OUTPUT_DIR'], exist_ok=True)
print(f"[ИНФО] Папка результатов: {COMPARE_CONFIG['OUTPUT_DIR']}")

### Шаг 4. Загрузка весов модели

In [ ]:
def load_model(config: dict, device: torch.device) -> nn.Module:
    """Загружает обученную модель."""
    model = ClutterFilter3DUNet(
        in_channels=config['IN_CHANNELS'],
        out_channels=config['OUT_CHANNELS'],
        base_features=config['BASE_FEATURES'],
    ).to(device)

    ckpt_dir    = config['CHECKPOINTS_DIR']
    best_path   = os.path.join(ckpt_dir, 'model_best.pth')
    latest_path = os.path.join(ckpt_dir, 'checkpoint_latest.pth')

    path_to_load = None
    if   os.path.exists(best_path):   path_to_load = best_path
    elif os.path.exists(latest_path): path_to_load = latest_path

    if path_to_load:
        ckpt  = torch.load(path_to_load, map_location=device)
        state = ckpt.get('model_state_dict', ckpt)
        model.load_state_dict(state)
        epoch = ckpt.get('epoch', '?')
        print(f"[ИНФО] Загружены веса: {os.path.basename(path_to_load)} (epoch={epoch})")
    else:
        print("[ПРЕДУПРЕЖДЕНИЕ] Чекпоинт не найден — используется случайная инициализация.")

    model.eval()
    return model

### Шаг 5. Конвертация форматов данных

In [ ]:
def tensor_to_complex_np(t2xy: torch.Tensor) -> np.ndarray:
    """PyTorch [2, T, X, Y] → NumPy complex [T, X, Y]."""
    arr = t2xy.cpu().numpy()        # [2, T, X, Y], float32
    return arr[0] + 1j * arr[1]    # [T, X, Y], complex64


def complex_np_to_tensor(z: np.ndarray, dev) -> torch.Tensor:
    """NumPy complex [T, X, Y] → PyTorch [2, T, X, Y] на целевом устройстве."""
    iq = np.stack([z.real, z.imag], axis=0).astype(np.float32)  # [2, T, X, Y]
    return torch.from_numpy(iq).float().to(dev)

### Шаг 6. Классические фильтры

In [ ]:
def apply_fir_filter(z: np.ndarray, config: dict) -> np.ndarray:
    """КИХ-фильтр высоких частот (FIR High-Pass)."""
    N = config['FIR_ORDER']
    nc = 2.0 * config['FILTER_CUTOFF_RATIO']
    b = signal_proc.firwin(N + 1, cutoff=nc, pass_zero=False, window='hamming')
    I_f = signal_proc.lfilter(b, [1.0], z.real, axis=0)
    Q_f = signal_proc.lfilter(b, [1.0], z.imag, axis=0)
    return I_f + 1j * Q_f


def apply_iir_filter(z: np.ndarray, config: dict) -> np.ndarray:
    """БИХ-фильтр высоких частот (IIR Butterworth High-Pass)."""
    nc    = 2.0 * config['FILTER_CUTOFF_RATIO']
    b, a  = signal_proc.butter(config['IIR_ORDER'], nc, btype='high')

    T = z.shape[0]
    default_padlen = 3 * max(len(a), len(b))
    safe_padlen    = min(default_padlen, T // 2 - 1)
    safe_padlen    = max(safe_padlen, 1)

    I_f = signal_proc.filtfilt(b, a, z.real, axis=0, padtype='even', padlen=safe_padlen)
    Q_f = signal_proc.filtfilt(b, a, z.imag, axis=0, padtype='even', padlen=safe_padlen)
    return I_f + 1j * Q_f


def apply_svd_filter(z: np.ndarray, config: dict) -> np.ndarray:
    """SVD-фильтр клаттера (Casorati-SVD)."""
    r        = config['SVD_RANK']
    T, X, Y  = z.shape
    Z = z.reshape(T, X * Y).T           # [X*Y, T]
    U, S, Vh = np.linalg.svd(Z, full_matrices=False)
    S_filt      = S.copy()
    S_filt[:r]  = 0.0
    Z_filt = (U * S_filt[np.newaxis, :]) @ Vh
    return Z_filt.T.reshape(T, X, Y)

### Шаг 7. Метрики

In [ ]:
def derive_masks(y: torch.Tensor, threshold_db: float = -35.0):
    """Строит бинарные маски из эталонного тензора таргета."""
    power_map = torch.mean(y[0]**2 + y[1]**2, dim=0)
    power_db  = 10.0 * torch.log10(power_map + 1e-12)
    peak_db   = power_db.max()
    mask_v    = (power_db >= (peak_db + threshold_db)).float()
    return mask_v, (1.0 - mask_v)


def calculate_cs(x: torch.Tensor, z_out: torch.Tensor,
                 mask_clutter: torch.Tensor, eps: float = 1e-12) -> float:
    """CS — Clutter Suppression (Эффективность подавления клаттера), дБ."""
    x     = x.float();     z_out = z_out.float()
    mask  = mask_clutter.float()
    n     = mask.sum() + eps

    p_in  = torch.mean(x[0]**2    + x[1]**2,    dim=0)   # [X, Y]
    p_out = torch.mean(z_out[0]**2 + z_out[1]**2, dim=0)

    P_in  = (p_in  * mask).sum() / n
    P_out = (p_out * mask).sum() / n

    return (10.0 * torch.log10(P_in / (P_out + eps) + eps)).item()


def calculate_bsp(y: torch.Tensor, z_out: torch.Tensor,
                  mask_vessel: torch.Tensor, eps: float = 1e-12) -> float:
    """BSP — Blood Signal Preservation (Сохранность полезного сигнала), дБ."""
    y      = y.float();      z_out = z_out.float()
    mask   = mask_vessel.float().unsqueeze(0)   # [1, X, Y]
    err_sq = (z_out[0] - y[0])**2 + (z_out[1] - y[1])**2  # [T, X, Y]
    tgt_sq = y[0]**2 + y[1]**2                              # [T, X, Y]

    sum_err = (err_sq * mask).sum()
    sum_tgt = (tgt_sq * mask).sum()

    return (10.0 * torch.log10(sum_tgt / (sum_err + eps) + eps)).item()


def calculate_mve(y: torch.Tensor, z_out: torch.Tensor,
                  mask_vessel: torch.Tensor, eps: float = 1e-8) -> float:
    """MVE — Mean Velocity Error (Ошибка оценки средней скорости), рад."""
    y     = y.float();     z_out = z_out.float()
    mask  = mask_vessel.float()

    y_c   = torch.complex(y[0],     y[1])
    out_c = torch.complex(z_out[0], z_out[1])

    R_y   = torch.sum(y_c[1:]   * torch.conj(y_c[:-1]),   dim=0)
    R_out = torch.sum(out_c[1:] * torch.conj(out_c[:-1]), dim=0)

    V_y   = torch.angle(R_y)
    V_out = torch.angle(R_out)

    diff = torch.angle(torch.exp(1j * (V_out - V_y)))
    mve  = (torch.abs(diff) * mask).sum() / (mask.sum() + eps)
    return mve.item()

### Шаг 8. Сравнительный анализ

In [ ]:
def run_comparison(model: nn.Module, test_loader: DataLoader,
                   config: dict, device: torch.device) -> list:
    """Проводит пакетный инференс по тестовым кадрам."""
    methods  = ['3D U-Net', 'FIR', 'IIR', 'SVD']
    records  = []
    n_total  = len(test_loader)
    skipped  = 0

    print(f"\n{'='*70}")
    print(f"  СРАВНИТЕЛЬНОЕ ТЕСТИРОВАНИЕ  |  {n_total} сэмплов  |  {len(methods)} методов")
    print(f"{'='*70}")

    model.eval()
    use_amp = (device.type == 'cuda')

    with torch.no_grad():
        for idx, (xb, yb) in enumerate(test_loader):
            x = xb[0]   # [2, T, X, Y]
            y = yb[0]

            x_dev = x.to(device)
            y_dev = y.to(device)

            mask_v, mask_cl = derive_masks(y_dev, config['VESSEL_MASK_THRESHOLD_DB'])

            if mask_v.sum() < 8:
                skipped += 1
                continue

            # ---- Метод 1: 3D U-Net ----
            with torch.amp.autocast('cuda', enabled=use_amp):
                unet_out = model(x_dev.unsqueeze(0))[0].float()

            # ---- Конвертация входа для классики ----
            z_in = tensor_to_complex_np(x)

            # ---- Метод 2: FIR (КИХ) ----
            fir_out = complex_np_to_tensor(apply_fir_filter(z_in, config), device)

            # ---- Метод 3: IIR (БИХ Баттерворта) ----
            iir_out = complex_np_to_tensor(apply_iir_filter(z_in, config), device)

            # ---- Метод 4: SVD ----
            svd_out = complex_np_to_tensor(apply_svd_filter(z_in, config), device)

            outputs_map = {
                '3D U-Net': unet_out,
                'FIR':      fir_out,
                'IIR':      iir_out,
                'SVD':      svd_out,
            }

            for method, z_out in outputs_map.items():
                records.append({
                    'Sample':   idx,
                    'Method':   method,
                    'CS_dB':    calculate_cs(x_dev, z_out, mask_cl),
                    'BSP_dB':   calculate_bsp(y_dev, z_out, mask_v),
                    'MVE_rad':  calculate_mve(y_dev, z_out, mask_v),
                })

            step = max(1, n_total // 10)
            if (idx + 1) % step == 0:
                done = (idx + 1) - skipped
                print(f"  [{idx+1:4d}/{n_total}]  Обработано: {done} сэмплов")

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    n_done = (len(records) // len(methods))
    print(f"\n[ГОТОВО] Обработано: {n_done} сэмплов  |  Пропущено: {skipped}")
    return records

### Шаг 9. Итоговый отчет

In [ ]:
def build_report(records: list):
    """Строит сводный DataFrame с метриками M±SD."""
    df_full = pd.DataFrame(records)

    METHOD_ORDER = ['3D U-Net', 'FIR', 'IIR', 'SVD']
    METRICS = {
        'CS_dB':   'CS (дБ)↑',
        'BSP_dB':  'BSP (дБ)↑',
        'MVE_rad': 'MVE (рад)↓',
    }

    rows = []
    for m in METHOD_ORDER:
        sub  = df_full[df_full['Method'] == m]
        row  = {'Метод': m, 'N сэмплов': len(sub)}
        for col, label in METRICS.items():
            mu  = sub[col].mean()
            std = sub[col].std(ddof=1)
            row[label] = f"{mu:.3f} ± {std:.3f}"
        rows.append(row)

    df_summary = pd.DataFrame(rows).set_index('Метод')

    sep = "=" * 72
    print(f"\n{sep}")
    print("         СВОДНАЯ ТАБЛИЦА СРАВНИТЕЛЬНОГО АНАЛИЗА  (M ± SD)")
    print(f"{sep}")
    print(df_summary.to_string())
    print(f"{sep}")
    print("  Стрелки: ↑ — чем больше, тем лучше | ↓ — чем меньше, тем лучше")
    print(f"{sep}\n")

    return df_full, df_summary

In [ ]:

def plot_boxplots(df_full: pd.DataFrame, config: dict):
    """Строит боксплоты распределения CS, BSP, MVE."""
    METHOD_ORDER = ['3D U-Net', 'FIR', 'IIR', 'SVD']
    PALETTE = {
        '3D U-Net': '#1565C0',
        'FIR':      '#E65100',
        'IIR':      '#2E7D32',
        'SVD':      '#6A1B9A',
    }
    METRICS = [
        ('CS_dB',   'Clutter Suppression (CS), дБ',  '↑ Подавление клаттера'),
        ('BSP_dB',  'Blood Preservation (BSP), дБ',  '↑ Сохранность крови'),
        ('MVE_rad', 'Mean Velocity Error (MVE), рад', '↓ Ошибка скорости'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(17, 6))
    fig.suptitle(
        'Сравнительный анализ фильтрации клаттера\n'
        '3D U-Net  vs  FIR (КИХ)  vs  IIR (БИХ)  vs  SVD',
        fontsize=13, fontweight='bold', y=1.02
    )

    for ax, (col, ylabel, note) in zip(axes, METRICS):
        data    = [df_full[df_full['Method'] == m][col].dropna().values
                   for m in METHOD_ORDER]
        bp      = ax.boxplot(
            data,
            labels=METHOD_ORDER,
            patch_artist=True,
            widths=0.55,
            notch=False,
            medianprops=dict(color='black', linewidth=2.2),
            flierprops=dict(marker='o', markersize=3.5, alpha=0.5, markeredgewidth=0.5),
            whiskerprops=dict(linewidth=1.2),
            capprops=dict(linewidth=1.2),
        )
        for patch, mname in zip(bp['boxes'], METHOD_ORDER):
            patch.set_facecolor(PALETTE[mname])
            patch.set_alpha(0.72)

        ax.set_title(note, fontsize=11, pad=6)
        ax.set_ylabel(ylabel, fontsize=9.5)
        ax.set_xlabel('Метод', fontsize=9.5)
        ax.grid(axis='y', linestyle=':', alpha=0.55)
        ax.tick_params(axis='x', labelsize=9, rotation=10)

    legend_handles = [
        Patch(facecolor=PALETTE[m], alpha=0.72, label=m) for m in METHOD_ORDER
    ]
    fig.legend(
        handles=legend_handles, loc='lower center', ncol=4,
        bbox_to_anchor=(0.5, -0.07), fontsize=10, framealpha=0.9
    )

    plt.tight_layout()
    save_path = os.path.join(config['OUTPUT_DIR'], 'boxplots_comparison.png')
    plt.savefig(save_path, dpi=config['FIGURE_DPI'], bbox_inches='tight')
    print(f"[ИНФО] Боксплоты сохранены: {save_path}")

    matplotlib.use('module://matplotlib_inline.backend_inline')
    fig.show()
    plt.close(fig)

In [ ]:
def _power_db(t: torch.Tensor) -> np.ndarray:
    return 10.0 * np.log10(
        torch.mean(t[0].float()**2 + t[1].float()**2, dim=0).cpu().numpy() + 1e-8
    )


def _kasai_map(t: torch.Tensor, power_ref_db: np.ndarray,
               threshold_db: float = -15.0) -> np.ndarray:
    z_c = torch.complex(t[0].float(), t[1].float())
    R   = torch.sum(z_c[1:] * torch.conj(z_c[:-1]), dim=0).cpu().numpy()
    V   = np.angle(R)
    p   = _power_db(t)
    V[p - np.max(p) < threshold_db] = np.nan
    return V


def plot_median_dashboard(model: nn.Module, test_loader: DataLoader,
                          df_full: pd.DataFrame, config: dict,
                          device: torch.device):
    """Строит 8-панельный дашборд медианного сэмпла по BSP U-Net."""
    df_u    = df_full[df_full['Method'] == '3D U-Net'].copy().reset_index(drop=True)
    bsp_arr = df_u['BSP_dB'].values
    median_bsp  = np.median(bsp_arr)
    med_row_idx = int(np.argmin(np.abs(bsp_arr - median_bsp)))
    med_sample  = int(df_u.iloc[med_row_idx]['Sample'])

    print(f"\n[ДАШБОРД] Медианный сэмпл: #{med_sample} (BSP U-Net = {bsp_arr[med_row_idx]:.2f} дБ)")

    x_vis = y_vis = None
    for i, (xb, yb) in enumerate(test_loader):
        if i == med_sample:
            x_vis, y_vis = xb[0], yb[0]
            break

    if x_vis is None:
        print("[ПРЕДУПРЕЖДЕНИЕ] Медианный сэмпл не найден — пропускаем дашборд.")
        return

    x_dev = x_vis.to(device)

    model.eval()
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        unet_out = model(x_dev.unsqueeze(0))[0].float().cpu()

    z_in    = tensor_to_complex_np(x_vis)
    fir_t   = complex_np_to_tensor(apply_fir_filter(z_in, config), 'cpu')
    iir_t   = complex_np_to_tensor(apply_iir_filter(z_in, config), 'cpu')
    svd_t   = complex_np_to_tensor(apply_svd_filter(z_in, config), 'cpu')

    PANELS = {
        'Вход':        (x_vis.cpu(),   '#607D8B'),
        'Эталон':      (y_vis.cpu(),   '#388E3C'),
        '3D U-Net':    (unet_out,      '#1565C0'),
        'FIR (КИХ)':  (fir_t,         '#E65100'),
        'IIR (БИХ)':  (iir_t,         '#2E7D32'),
        'SVD':         (svd_t,         '#6A1B9A'),
    }

    n_cols = len(PANELS)
    fig    = plt.figure(figsize=(3.8 * n_cols, 9))
    gs     = gridspec.GridSpec(2, n_cols, figure=fig, hspace=0.30, wspace=0.10)
    fig.suptitle(
        f'Дашборд медианного сэмпла #{med_sample}  |  Сравнение методов фильтрации клаттера',
        fontsize=12, fontweight='bold', y=1.01
    )

    p_ref = np.max(_power_db(x_vis.cpu()))

    for col_idx, (label, (tensor, color)) in enumerate(PANELS.items()):
        tensor = tensor.cpu().float()

        # ── Строка 1: Power Doppler ──
        ax_top = fig.add_subplot(gs[0, col_idx])
        p_map  = _power_db(tensor) - p_ref
        im_pd  = ax_top.imshow(p_map.T, cmap='viridis', vmin=-45, vmax=0, origin='upper', aspect='auto')
        ax_top.set_title(label, fontsize=9.5, color=color, fontweight='bold', pad=4)
        if col_idx == 0:
            ax_top.set_ylabel('Глубина Y', fontsize=8)
        else:
            ax_top.set_yticks([])
        ax_top.set_xlabel('Луч X', fontsize=7.5)
        ax_top.tick_params(labelsize=7)

        # ── Строка 2: Color Doppler ──
        ax_bot = fig.add_subplot(gs[1, col_idx])
        V_map  = _kasai_map(tensor, p_ref)
        cmap_d = plt.cm.RdBu_r.copy()
        cmap_d.set_bad(color='#EEEEEE')
        ax_bot.imshow(V_map.T, cmap=cmap_d, vmin=-np.pi, vmax=np.pi, origin='upper', aspect='auto')
        if col_idx == 0:
            ax_bot.set_ylabel('Глубина Y', fontsize=8)
        else:
            ax_bot.set_yticks([])
        ax_bot.set_xlabel('Луч X', fontsize=7.5)
        ax_bot.tick_params(labelsize=7)

    # Цветовые шкалы
    ax_cb1 = fig.add_axes([0.92, 0.54, 0.013, 0.36])
    plt.colorbar(plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(-45, 0)), cax=ax_cb1, label='Мощность (дБ)')
    ax_cb1.tick_params(labelsize=7)

    ax_cb2 = fig.add_axes([0.92, 0.10, 0.013, 0.36])
    plt.colorbar(plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(-np.pi, np.pi)), cax=ax_cb2, label='Фаза (рад)')
    ax_cb2.tick_params(labelsize=7)

    fig.text(0.005, 0.72, 'Power Doppler\n(Energy Map)', va='center', ha='left', rotation=90, fontsize=9, fontweight='bold', color='#37474F')
    fig.text(0.005, 0.28, 'Color Doppler\n(Kasai Velocity)', va='center', ha='left', rotation=90, fontsize=9, fontweight='bold', color='#37474F')

    save_path = os.path.join(config['OUTPUT_DIR'], 'dashboard_median_sample.png')
    plt.savefig(save_path, dpi=config['FIGURE_DPI'], bbox_inches='tight')
    print(f"[ИНФО] Дашборд медианного сэмпла сохранён: {save_path}")

    matplotlib.use('module://matplotlib_inline.backend_inline')
    fig.show()
    plt.close(fig)

    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
def save_reports(df_full: pd.DataFrame, df_summary: pd.DataFrame, config: dict):
    """Сохраняет полный и сводный DataFrame в CSV-файлы на Google Drive."""
    path_full    = os.path.join(config['OUTPUT_DIR'], 'comparison_full.csv')
    path_summary = os.path.join(config['OUTPUT_DIR'], 'comparison_summary.csv')

    df_full.to_csv(path_full,    index=False, encoding='utf-8-sig')
    df_summary.to_csv(path_summary,           encoding='utf-8-sig')

    print(f"[ИНФО] Полные данные сохранены:  {path_full}")
    print(f"[ИНФО] Сводная таблица:          {path_summary}")

In [ ]:
def main():
    print("\n" + "=" * 70)
    print("  АВТОНОМНЫЙ БЛОК СРАВНИТЕЛЬНОГО АНАЛИЗА ФИЛЬТРАЦИИ КЛАТТЕРА")
    print("  Методы: 3D U-Net  |  FIR (КИХ)  |  IIR (БИХ)  |  SVD")
    print("=" * 70)

    # Шаг 1: Тестовый DataLoader
    test_loader = build_test_loader(COMPARE_CONFIG)

    # Шаг 2: Загрузка обученной модели
    model = load_model(COMPARE_CONFIG, device)

    # Шаг 3: Сравнительный анализ
    records = run_comparison(model, test_loader, COMPARE_CONFIG, device)

    if not records:
        print("\n[ОШИБКА] Нет данных для анализа.")
        return

    # Шаг 4: Отчёт M±SD
    df_full, df_summary = build_report(records)

    # Шаг 5: Боксплоты
    plot_boxplots(df_full, COMPARE_CONFIG)

    # Шаг 6: Дашборд медианного сэмпла
    plot_median_dashboard(model, test_loader, df_full, COMPARE_CONFIG, device)

    # Шаг 7: Сохранение CSV
    save_reports(df_full, df_summary, COMPARE_CONFIG)

    print("\n" + "=" * 70)
    print("[ЗАВЕРШЕНО] Сравнительный анализ успешно выполнен.")
    print(f"[РЕЗУЛЬТАТЫ] {COMPARE_CONFIG['OUTPUT_DIR']}")
    print("=" * 70)

In [ ]:
main()